In [1]:
import torch
from torch.utils.data import DataLoader, random_split, TensorDataset
import logging
from torch import nn 

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
import urllib.request

# Download Tiny Shakespeare directly into your workspace
# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
file_path = "tinyshakespeare.txt"

# urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text)} characters")
print("Sample text:\n", text[:150])

Dataset length: 1115394 characters
Sample text:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


In [3]:
itos, stoi = dict(), dict() 

words = text.split(' ')
unique_words = sorted(list(set(text.split(' '))))
vocab_size = len(unique_words)

# stoi and itos mappings 
stoi = {w: i for (i, w) in enumerate(unique_words)}
itos = {i: w for (i, w) in enumerate(unique_words)}

def tensor_to_sentence(token_tensor):
    if not isinstance(token_tensor, torch.Tensor):
        token_tensor = torch.tensor(token_tensor, dtype=torch.long)

    token_tensor = token_tensor.detach().cpu()

    if token_tensor.dim() == 0:
        return itos.get(int(token_tensor.item()), "<UNK>")

    if token_tensor.dim() == 1:
        return " ".join(itos.get(int(idx), "<UNK>") for idx in token_tensor.tolist())

    if token_tensor.dim() == 2:
        return [
            " ".join(itos.get(int(idx), "<UNK>") for idx in row.tolist())
            for row in token_tensor
        ]

    raise ValueError("Expected a 0D, 1D, or 2D tensor of token ids.")


data = torch.tensor([stoi[w] for w in words], dtype=torch.long)

# We need to create "chunks of text" 
seq_len = 64
n = data.shape[0]
X_list, Y_list = list(), list()
for i in range(0, len(data) - seq_len, seq_len):
    x = data[i: i + seq_len]
    y = data[i + 1: i + seq_len + 1]

    X_list.append(x)
    Y_list.append(y)


X = torch.stack(X_list, dim=0)
Y = torch.stack(Y_list, dim=0)


batch_size = 8
dataset = TensorDataset(X, Y)
train_size = int(.80*len(dataset))
test_size = int(.10*len(dataset))
val_size = len(dataset) - train_size - test_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, test_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  drop_last=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,  drop_last=True, pin_memory=True)



print(f"X tensor shape {X.shape}")
print(f"Y tensor shape {Y.shape}")
print(f"vocab size:  {vocab_size}")
assert torch.equal(X[:, 1:seq_len], Y[:, :seq_len - 1]), "Label != Next Token"

print(f"len(train_loader): {len(train_loader)}")
print(f"len(val_loader): {len(val_loader)}")
print(f"len(test_loader): {len(test_loader)}")



X tensor shape torch.Size([2654, 64])
Y tensor shape torch.Size([2654, 64])
vocab size:  42197
len(train_loader): 265
len(val_loader): 33
len(test_loader): 33


In [4]:
class RMSNorm(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.epsilon = 1e-6
        self.gamma = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.tensor):
        mean = torch.mean(torch.pow(x, 2),dim=-1, keepdim=True)
        rms = torch.sqrt(mean + self.epsilon)
        rms_norm = (x / rms) * self.gamma
        return rms_norm


norm = RMSNorm(2)
norm.epsilon = 0


x = torch.tensor(
    [
        [[1, 1], [2, 2], [3, 3], [4, 4]],
        [[5, 5], [6, 6], [7, 7], [8, 8]]
    ], dtype=torch.float
)

expected = torch.tensor(
    [
        [[1, 1], [1, 1], [1, 1], [1, 1]],
        [[1, 1], [1, 1], [1, 1], [1, 1]],
    ], dtype=torch.float
)

o = norm(x)
assert torch.equal(o, expected), f"{o} != {expected}"


In [ ]:

x = torch.tensor([[1, 1, 1, 1], [2, 2, 2, 2], [3, 3, 3, 3], [4, 4, 4, 4]], dtype=float)

seq_len, d_model = x.shape[0], x.shape[1]


# Notes: 
# m - This value controls how much we rotate the embedding and it is depedent on the token's
#     location in the sequence. Token's later in the sequency are rotated furhter. 
# theta - This value controls the frequency at which we rotate. This value is predetermined, 
#         and is depedent only on d_model and the RoPE base frequency (10k). Theta is calculated 
#         for d_model/2 pairs and pairs that are near the beginning of the embedding rotate 
#         have 


i = torch.arange(0, d_model, 2, dtype=torch.float)
theta = torch.pow(10000, -i/d_model)
m = torch.arange(0, seq_len)
rope_angles = m.unsqueeze(-1) * theta


cos = torch.cos(rope_angles)
sin = torch.sin(rope_angles)



x_grouped = x.view(4, 2, 2).unsqueeze(-1)

print(rope_angles.shape)




rotation = torch.tensor([[10, 10], [100, 100]], dtype=float)
x_rotated = (rotation @ x_grouped).squeeze(-1).view(4, -1)

# print(x_rotated)

# print(x_rotated.shape)

# We have input of x of size 4 tokens with dim = 4

# We need to rotate each token's embedding 










torch.Size([4, 2])


In [5]:


d_model = 8



for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
    embedding = nn.Embedding(vocab_size, d_model)

    break 